In [ ]:
# auto_stop_fulln.ipynb -- overnight sentinel. Run this on EVERY pod before
# sleeping. It watches the shared out_dir; once every FULL-N (n=2000) combo is
# terminal (done or failed), it SIGTERMs local training (the supervisor
# self-revokes its lease) and STOPS THIS RUNPOD POD -- billing ends, the
# /workspace volume persists. Small-n combos picked up during the drain tail
# are interrupted safely: checkpoints resume later and reconcile refunds the
# attempt. Morning plan: start 2-3 pods, run eval_champions -> prep-select ->
# prep-prune, then resume the champion tail on the cheap subset.
import copy, json, os, signal, subprocess, sys, time
from pathlib import Path

REPO = "/workspace/stable-query-latent"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"
POLL_SECONDS = 120
HARD_DEADLINE_HOURS = 12    # stop the pod after this many hours NO MATTER WHAT
                            # (safety net so a stuck combo can't bill all night)

if REPO not in sys.path:
    sys.path.insert(0, REPO)
from VICReg_review.sweep.config import SweepConfig

root = Path(REPO) / OUT_DIR
cfg = SweepConfig.load(f'{REPO}/VICReg_review/sweep/sweep.yaml')
combos = list(cfg.iter_combos())
_counts = [int(c.train_games) for c in combos]
FULL_N = 0 if any(n <= 0 for n in _counts) else max(_counts)
fulln_ids = [c.combo_id for c in combos if int(c.train_games) == FULL_N]
pod_id = os.environ.get('RUNPOD_POD_ID')
print(f'sentinel: {len(fulln_ids)} FULL-N combos to watch; pod={pod_id}; '
      f'poll={POLL_SECONDS}s; hard deadline={HARD_DEADLINE_HOURS}h')
if not pod_id:
    print('!! RUNPOD_POD_ID not set -- will stop training but CANNOT stop the pod;')
    print('   it would keep billing while idle. Check the env before relying on this.')


def _terminal(cid):
    d = root / cid
    if (d / 'done.json').exists() or (d / 'failed.json').exists():
        return True
    try:
        return json.loads((d / 'vicreg_review_h5_manifest.json')
                          .read_text(encoding='utf-8')).get('status') == 'done'
    except Exception:
        return False


def remaining():
    return [cid for cid in fulln_ids if not _terminal(cid)]


def stop_everything(reason):
    print(f'sentinel: {reason} -- stopping training on this pod', flush=True)
    subprocess.run(['pkill', '-f', 'sweep/supervisor.py'])   # SIGTERM: lease self-revokes
    time.sleep(10)
    subprocess.run(['pkill', '-9', '-f', 'sweep/worker.py'])
    if pod_id:
        print(f'sentinel: runpodctl stop pod {pod_id}', flush=True)
        r = subprocess.run(['runpodctl', 'stop', 'pod', pod_id],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr, flush=True)
        if r.returncode != 0:
            print('!! runpodctl stop FAILED -- pod is idle but still billing; '
                  'stop it from the RunPod console.', flush=True)
    else:
        print('!! no pod id: training stopped, but STOP THE POD from the console.',
              flush=True)


deadline = time.time() + HARD_DEADLINE_HOURS * 3600
while True:
    left = remaining()
    if not left:
        stop_everything('FULL-N group fully terminal')
        break
    if time.time() >= deadline:
        stop_everything(f'hard deadline {HARD_DEADLINE_HOURS}h reached '
                        f'({len(left)} FULL-N combos still open)')
        break
    print(f'{time.strftime("%H:%M:%S")} FULL-N remaining: {len(left)}'
          + (f'  (next few: {", ".join(left[:3])})' if len(left) <= 6 else ''),
          flush=True)
    time.sleep(POLL_SECONDS)
